# 04 Gold: the star schema

Reads the clean Silver table (`data/silver/readings.parquet`) and the reference tables from Bronze,
and builds a **star schema** in `data/gold`: one fact table in the middle, four dimension tables around it,
plus one daily summary. Picture of the plan: https://claude.ai/artifact/SEbGL8593WUgz2qUN5T4qM

| Table | Rows × columns | One row is |
| --- | --- | --- |
| `dim_state` | 3 × 3 | a state: ON, OFF, ERROR |
| `dim_station` | 7 × 5 | a solar plant |
| `dim_device` | 66 × 6 | a device (53 are inverters) |
| `dim_date` | 340 × 7 | a calendar day |
| `fact_readings` | 1,180,215 × 13 | one reading |
| `fact_daily_energy` | 8,248 × 4 | one inverter on one day |

**Fact** = long and narrow, the numbers (measurements) plus keys. **Dimension** = short and wide, the words you filter and group by.

## 0. Setup

Load Silver. Expected shape: (1180215, 19): the 16 Silver columns, the common state (2), and the `bad_energy_today` flag.

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BRONZE = ROOT / "data" / "bronze"
SILVER = ROOT / "data" / "silver"
GOLD = ROOT / "data" / "gold"
GOLD.mkdir(exist_ok=True)

readings = pd.read_parquet(SILVER / "readings.parquet")
print(readings.shape)

(1180215, 19)


## 1. dim_state

The smallest dimension: one row per state, and a key (`canonical_state_id`) the fact can point to.

In [3]:
dim_state = pd.read_parquet(BRONZE / "inverter_state_canonical.parquet")
dim_state = dim_state[["canonical_state_id", "description"]]
dim_state["state"] = dim_state["canonical_state_id"].map({"1": "ON", "2": "OFF", "3": "ERROR"})
print(dim_state)

  canonical_state_id                                        description  state
0                  1                     ON: producing / grid-connected     ON
1                  2  OFF: standby / offline / no irradiation / not ...    OFF
2                  3  ERROR: alarm / fault / shutdown abnormal / com...  ERROR


## 2. dim_station and dim_device

Both reference tables store the vendor as a number in `source`. The tiny `source` table becomes a lookup
(`{'1': 'solis', '2': 'fusionsolar'}`) instead of its own dimension.

In [4]:
source = pd.read_parquet(BRONZE / "source.parquet")
vendor = dict(zip(source["id"], source["name"].str.lower()))
print(vendor)

{'1': 'solis', '2': 'fusionsolar'}


**dim_station:** 7 plants, 1 Solis and 6 FusionSolar. Text columns from Bronze become numbers.

In [5]:
stations = pd.read_parquet(BRONZE / "stations.parquet")
dim_station = stations[["station_hash_id", "name", "location_latitude", "plant_age_years"]].copy()
dim_station["vendor"] = stations["source"].map(vendor)
dim_station = dim_station.rename(columns={"location_latitude": "latitude"})
dim_station["latitude"] = pd.to_numeric(dim_station["latitude"])
dim_station["plant_age_years"] = pd.to_numeric(dim_station["plant_age_years"])
print(dim_station)

                    station_hash_id                   name  latitude  \
0  2c85da17f3ac504327da04874ef1a701        SOLIS Station 1  41.86956   
1  e2afd8af435a86d8bfe72dcbe937a5d6  FUSIONSOLAR Station 1  42.00973   
2  19477f81a2058c850a09310b7795fd4f  FUSIONSOLAR Station 2  41.14166   
3  ae68bd65b1c87d7a5985d1d5498442af  FUSIONSOLAR Station 3  41.14166   
4  55cddba5799e63f231eae0eb8cca8247  FUSIONSOLAR Station 4  41.14166   
5  b59685487a1fc5712e03c264f70cb656  FUSIONSOLAR Station 5  42.00973   
6  7534daaee746a48bc5b99132810f3e69  FUSIONSOLAR Station 6  42.00973   

   plant_age_years       vendor  
0              1.5        solis  
1              1.5  fusionsolar  
2              1.5  fusionsolar  
3              1.5  fusionsolar  
4              1.5  fusionsolar  
5              1.5  fusionsolar  
6              1.5  fusionsolar  


**dim_device:** 66 devices, but only **53 are inverters**. The other 13 (power meters, Smart Loggers, dongles) have an empty
`max_power`, which is why `errors="coerce"` is safe here. They never send readings.
`station_hash_id` stays in the table: it's how a reading finds its plant.

In [6]:
devices = pd.read_parquet(BRONZE / "devices.parquet")
dim_device = devices[["device_hash_id", "station_hash_id", "name", "device_model", "max_power"]].copy()
dim_device["vendor"] = devices["source"].map(vendor)
dim_device = dim_device.rename(columns={"max_power": "max_power_kw"})
dim_device["max_power_kw"] = pd.to_numeric(dim_device["max_power_kw"], errors="coerce")
print(dim_device.shape)
print(dim_device["device_model"].value_counts())

(66, 6)
device_model
SUN2000-50KTL-M3    33
1603                13
SUN2000-40KTL-M3     7
PowerMeter           6
Smart Logger         4
SDongleA-05          2
                     1
Name: count, dtype: int64


## 3. dim_date

One row per calendar day, from the first reading to the last, **including days without readings**, so a missing day shows up
as empty in a report instead of disappearing. Power BI calls this a date table.
`date_key` = 20240627 style: readable and quick to join. The UTC date is safe for solar data: it's dark at midnight.

In [7]:
first = readings["ts"].min().date()
last = readings["ts"].max().date()
dim_date = pd.DataFrame({"date": pd.date_range(first, last, freq="D")})

dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["weekday"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["date"].dt.day_of_week >= 5

print(dim_date.shape)
print(dim_date.head(3))

(340, 7)
        date  date_key  year  month month_name   weekday  is_weekend
0 2024-06-27  20240627  2024      6       June  Thursday       False
1 2024-06-28  20240628  2024      6       June    Friday       False
2 2024-06-29  20240629  2024      6       June  Saturday        True


## 4. fact_readings

Each reading picks up its plant from `dim_device`, gets a `date_key`, and the six Silver flags collapse into one `is_valid`
(True when no flag is raised). The fact keeps only keys and measurements: 13 columns.

Expected: is_valid True 1,177,533 / False 2,682, and 276 readings without a station (the made-up device id).

In [8]:
fact = readings.merge(dim_device[["device_hash_id", "station_hash_id"]], on="device_hash_id", how="left")
fact["date_key"] = fact["ts"].dt.strftime("%Y%m%d").astype(int)

FLAGS = ["unknown_device", "bad_temperature", "bad_counter", "unknown_state", "too_much_power", "bad_energy_today"]
fact["is_valid"] = ~fact[FLAGS].any(axis=1)

fact = fact[["date_key", "device_hash_id", "station_hash_id", "canonical_state_id", "ts",
             "ac_power_kw", "dc_power_kw", "efficiency_pct", "energy_today_kwh",
             "energy_total_kwh", "temperature_c", "is_valid", "source_file"]]

print(fact.shape)
print(fact["is_valid"].value_counts())
print(fact["station_hash_id"].isna().sum())

(1180215, 13)
is_valid
True     1177533
False       2682
Name: count, dtype: int64
276


A look at 20 daytime readings (`ac_power_kw > 0`). A table on the last line, without `print`, is drawn as a formatted table.

In [9]:
pd.set_option("display.max_columns", None)
fact[fact["ac_power_kw"] > 0].head(20)

,date_key,device_hash_id,station_hash_id,canonical_state_id,ts,ac_power_kw,dc_power_kw,efficiency_pct,energy_today_kwh,energy_total_kwh,temperature_c,is_valid,source_file
24,20250101,94aedd81bad035cc73968fe5b06f808a,19477f81a2058c850a09310b7795fd4f,1,2025-01-01 06:50:00+00:00,0.947,1.068675,88.19,0.43,48316.57,26.7,True,fusionsolar_2025-01-01.csv
25,20250101,94aedd81bad035cc73968fe5b06f808a,19477f81a2058c850a09310b7795fd4f,1,2025-01-01 08:40:00+00:00,9.497,9.682288,98.14,9.09,48325.23,36.9,True,fusionsolar_2025-01-01.csv
26,20250101,484582ba5d7642fa214a5d6e3b115b0a,b59685487a1fc5712e03c264f70cb656,1,2025-01-01 07:30:00+00:00,0.606,0.591391,80.23,0.85,88764.33,23.2,True,fusionsolar_2025-01-01.csv
28,20250101,6773fffa87bd4b069c5be8cf15da35f4,7534daaee746a48bc5b99132810f3e69,1,2025-01-01 10:10:00+00:00,0.110,0.404796,49.60,0.16,84392.26,24.3,True,fusionsolar_2025-01-01.csv
30,20250101,e1064be3bcf17f4b2319774987bb4365,55cddba5799e63f231eae0eb8cca8247,1,2025-01-01 08:30:00+00:00,8.615,8.670815,97.13,8.56,93645.82,35.3,True,fusionsolar_2025-01-01.csv
31,20250101,c65473cc38070d9a622e063b9083eb43,ae68bd65b1c87d7a5985d1d5498442af,1,2025-01-01 09:40:00+00:00,16.485,16.891364,97.91,27.94,74727.90,32.3,True,fusionsolar_2025-01-01.csv
32,20250101,94aedd81bad035cc73968fe5b06f808a,19477f81a2058c850a09310b7795fd4f,1,2025-01-01 08:45:00+00:00,10.154,10.350600,98.25,9.90,48326.04,37.4,True,fusionsolar_2025-01-01.csv
34,20250101,6187ec08ac3e790c94c9ae81b0af5e1c,7534daaee746a48bc5b99132810f3e69,1,2025-01-01 09:20:00+00:00,0.109,0.000002,49.60,0.11,85908.26,30.7,True,fusionsolar_2025-01-01.csv
36,20250101,6187ec08ac3e790c94c9ae81b0af5e1c,7534daaee746a48bc5b99132810f3e69,1,2025-01-01 06:45:00+00:00,0.002,0.000002,49.60,0.00,85908.15,16.3,True,fusionsolar_2025-01-01.csv
37,20250101,23c1064d4afe10e3af7fff7bd9c2a227,b59685487a1fc5712e03c264f70cb656,1,2025-01-01 08:00:00+00:00,0.989,1.047970,83.53,1.24,88364.99,28.8,True,fusionsolar_2025-01-01.csv


## 5. fact_daily_energy

`energy_today_kwh` is a counter that starts at 0 each morning: the inverter integrates power itself (E = ∫P dt).
So a day's energy is its **highest value**, which is more accurate than adding up power × Δt from 5-minute samples.

**Gold found a problem Silver missed.** The first version had a max of 39,086,239 kWh for one inverter-day, while
55 kW × 24 h = 1,320 kWh is the physical limit. One reading in the original data (6 Jan 2025 13:45, power −1.7 million kW).
The fix went into **Silver** (flag `bad_energy_today`), not here. Now the max is 968 kWh.

In [10]:
good = fact[fact["is_valid"]]
daily = good.groupby(["date_key", "station_hash_id", "device_hash_id"], as_index=False)["energy_today_kwh"].max()
daily = daily.rename(columns={"energy_today_kwh": "energy_kwh"})

print(daily.shape)
print(daily["energy_kwh"].describe())

(8248, 4)
count    8248.000000
mean      266.917478
std       263.637062
min         0.000000
25%        60.687500
50%       182.305000
75%       404.500000
max       968.100000
Name: energy_kwh, dtype: float64


## 6. Using the star: energy per plant per month

Two joins and one `groupby`: the daily table gives the numbers, `dim_station` the plant name, `dim_date` the month.
Check: SOLIS Station 1, May 2025 = 271,033 kWh. The values rise from winter to May, following the sun.

In [11]:
report = daily.merge(dim_station[["station_hash_id", "name"]], on="station_hash_id")
report = report.merge(dim_date[["date_key", "year", "month"]], on="date_key")

monthly = report.groupby(["name", "year", "month"])["energy_kwh"].sum().round(0)
print(monthly.to_string())

name                   year  month
FUSIONSOLAR Station 1  2024  12            0.0
                       2025  1          1409.0
                             2         14453.0
                             3         24594.0
                             4         42132.0
                             5         63372.0
FUSIONSOLAR Station 2  2024  12            0.0
                       2025  1          1193.0
                             2          1861.0
                             3          1200.0
                             4          3293.0
                             5          5402.0
                             6             0.0
FUSIONSOLAR Station 3  2025  1          1764.0
                             2          2582.0
                             3          1509.0
                             4          3959.0
                             5          6110.0
FUSIONSOLAR Station 4  2024  12            0.0
                       2025  1          7782.0
                         

## 7. Save

All six tables as Parquet in `data/gold`. In Fabric these become Delta tables, and Power BI draws the same relationships.

In [12]:
tables = {
    "dim_date": dim_date,
    "dim_device": dim_device,
    "dim_station": dim_station,
    "dim_state": dim_state,
    "fact_readings": fact,
    "fact_daily_energy": daily,
}

for name, table in tables.items():
    table.to_parquet(GOLD / f"{name}.parquet", index=False)
    print(name, table.shape)

dim_date (340, 7)
dim_device (66, 6)
dim_station (7, 5)
dim_state (3, 3)
fact_readings (1180215, 13)
fact_daily_energy (8248, 4)


## Result

The local Medallion is complete: Landing → Bronze → Silver → Gold.
Next: rebuild it in Microsoft Fabric (Lakehouse, Delta tables, Spark, SQL) and finish with a Power BI semantic model.